In [1]:

%pip install -r requirements-bilbo.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-bilbo.txt'


In [4]:
!pip install -q gensim wordninja joblib scikit-learn tqdm tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 8.9 MB/s eta 0:00:00


In [5]:
import string
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.nn as nn
import tldextract
import torch.optim as optim
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)
from torch.utils.data import Dataset, DataLoader

In [6]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Bilbo_ver2

Mounted at /content/drive
/content/drive/MyDrive/Bilbo_ver2


In [7]:
train_df = pd.read_csv('data/train_genAI.csv')
val_df = pd.read_csv('data/val_genAI.csv')
test_df = pd.read_csv('data/test_genAI.csv')
print(train_df.head(5))

   Unnamed: 0          domain_name subclass  label
0           0     6301un092rsh.org   qadars      1
1           1        samruk-oil.kz   benign      0
2           2  9pi4r5jdh1fi9g2.net  shiotob      1
3           3      capuaenc.com.ar   benign      0
4           4           nasphv.org   benign      0


In [8]:
import tldextract
import re
_NON_SLD_RE = re.compile(r"[^a-z0-9-]+")
def normalize_sld(domain: str) -> str:
    ext = tldextract.extract(domain)
    return _NON_SLD_RE.sub("", ext.domain.lower())

In [9]:

X_train = train_df['domain_name'].apply(normalize_sld).values
y_train = train_df['label'].values
X_val = val_df['domain_name'].apply(normalize_sld).values
y_val = val_df['label'].values
X_test = test_df['domain_name'].apply(normalize_sld).values
y_test = test_df['label'].values
print(X_test[:10])
print(len(X_train))

['carnegiesciencecenter' 'tadalafil-cheapestprice-canadian'
 'newburyracecourse' 'giocompany' 'barbarafamiliainsurance' 'abe70b78b996'
 'qpwharrres' 'purplestea' 'wh' 'shenduwang']
986316


In [10]:
def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

num_epochs    = 10
batch_size    = 64
lr            = 1e-3
weight_decay  = 1e-2
max_grad_norm = 1.0
MAX_LEN       = 32

ALPHABET  = string.ascii_lowercase + string.digits + '.' + '-'
char2idx  = {c: i+1 for i, c in enumerate(ALPHABET)}
idx2char  = {i: c for c, i in char2idx.items()}
vocab_size = len(char2idx) + 1
print(f"Vocab size: {vocab_size}")

def domain_to_char_tensor(domain: str) -> torch.Tensor:
    str_domain = str(domain)
    s   = str_domain[:MAX_LEN]
    arr = [char2idx.get(c, 0) for c in s] + [0]*(MAX_LEN - len(s))
    return torch.tensor(arr, dtype=torch.long)
g = domain_to_char_tensor(X_test[3])
print(X_train[6],g)


Device: cuda
Vocab size: 39
stellaconnect tensor([ 7,  9, 15,  3, 15, 13, 16,  1, 14, 25,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])


In [ ]:
class DomainDataset(Dataset):
    def __init__(self,domain,label):
        self.domain = np.asarray(domain)
        self.label = np.asarray(label,dtype=int)
    def __len__(self):
        return len(self.domain)
    def __getitem__(self, key):
        domain_tensor = domain_to_char_tensor(self.domain[key])
        y = torch.tensor(self.label[key],dtype=torch.long)
        return domain_tensor,y
train_dataset = DomainDataset(X_train,y_train)
val_dataset = DomainDataset(X_val,y_val)
test_dataset = DomainDataset(X_test,y_test)

kw = dict(num_workers=0, pin_memory=True)
# hoặc: kw={'num_workers': 0, 'pin_memory': True }
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True , num_workers=0,pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size,shuffle=False, num_workers=0,pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size,shuffle=False, num_workers=0,pin_memory=True)

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")

Train: 986,316 | Val: 123,290 | Test: 123,293


In [12]:
INPUT_SHAPE        = 40           # MAX_STRING_LENGTH
MAX_INDEX          = 10000        # vocabulary size  (input_dim of CNN embedding)
MAX_FEATURES       = 128          # output_dim of LSTM embedding
EMBEDDING_DIMENSION = 128         # output_dim of CNN embedding
NUM_CONV_FILTERS   = 128
LSTM_HIDDEN        = 256
class Hybrid_CNN_LSTM(nn.Module):
    def __init__(
        self,
        embedding_dim:int = EMBEDDING_DIMENSION,
        max_string_length:int = INPUT_SHAPE,
        num_filter = NUM_CONV_FILTERS,
        embedding_lstm = MAX_FEATURES,
        lstm_hidden = LSTM_HIDDEN
    ):
        super().__init__()
        # CNN branch
        self.embedding_cnn = nn.Embedding(num_embeddings=max_string_length,embedding_dim=embedding_dim,padding_idx=0)
        kernel_size = [2,3,4,5,6]
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filter,
                kernel_size=k
            )
            for k in kernel_size
        ])
        self.dropout = nn.Dropout(0.5)
        self.dense = nn.Linear(len(kernel_size)*num_filter,num_filter)

        # LSTM branch
        self.embedding_lstm = nn.Embedding(
            num_embeddings=max_string_length,
            embedding_dim=embedding_lstm,
            padding_idx=0
        )
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden,
            batch_first=True
        )
        combun_dim = num_filter + lstm_hidden
        self.extra_dense = nn.Linear(combun_dim,100)
        self.output_layer = nn.Linear(100,1)
    def forward(self,x:torch.tensor) -> torch.tensor:
        emb_cnn = self.embedding_cnn(x) # (B,L,emb_dim)
        emb_cnn = emb_cnn.permute(0,2,1)
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(emb_cnn))
            p = F.adaptive_max_pool1d(c,1).squeeze(-1)
            pooled.append(p)
        concat_cnn = torch.cat(pooled,dim=1)
        cnn_out = self.dropout(concat_cnn)
        cnn_out     = F.relu(self.dense(cnn_out))
        cnn_out     = self.dropout(cnn_out)      # (B, filters)
        # LSTM
        emb_lstm    = self.embedding_lstm(x)         # (B, L, 256)
        _, (h_n, _) = self.lstm(emb_lstm)            # h_n: (1, B, hidden)
        lstm_out    = h_n.squeeze(0)                 # (B, hidden)
        lstm_out    = self.dropout(lstm_out)

        # ── Combine ───────────────────────────────────────────────────
        combined    = torch.cat([cnn_out, lstm_out], dim=1)   # (B, 384)
        out         = self.dropout(combined)
        out         = F.relu(self.extra_dense(out))
        out         = self.dropout(out)
        out         = torch.sigmoid(self.output_layer(out))   # (B, 1)

        return out
model = Hybrid_CNN_LSTM().to(device)
print(model.state_dict(),model.parameters)


OrderedDict({'embedding_cnn.weight': tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 1.9312,  1.0119, -1.4364,  ...,  0.5655,  0.5058,  0.2225],
        [-0.6855,  0.5636, -1.5072,  ...,  0.8541, -0.4901, -0.3595],
        ...,
        [-0.3434,  0.9878,  1.3733,  ..., -1.6977, -1.2025, -1.6465],
        [-0.7897, -1.3139,  1.2118,  ...,  1.5608,  0.0496, -0.3332],
        [ 0.9818,  1.0984, -0.2567,  ..., -0.6820,  1.0070, -0.8279]],
       device='cuda:0'), 'convs.0.weight': tensor([[[-0.0130, -0.0611],
         [-0.0607, -0.0082],
         [ 0.0447, -0.0412],
         ...,
         [ 0.0560,  0.0128],
         [ 0.0423,  0.0506],
         [ 0.0491, -0.0570]],

        [[-0.0408, -0.0041],
         [-0.0002,  0.0053],
         [ 0.0525,  0.0566],
         ...,
         [ 0.0189,  0.0379],
         [ 0.0202, -0.0030],
         [ 0.0046,  0.0541]],

        [[ 0.0272, -0.0144],
         [-0.0502,  0.0456],
         [-0.0233,  0.0390],
         ...,
     

In [13]:
import time
n0 = int((y_train == 0).sum())
n1 = int((y_train == 1).sum())
s = (n0 / max(1,n1)) *1.2
pos_weight = torch.tensor([s], dtype=torch.float32).to(device)
print(f"Benign: {n0:,} | DGA: {n1:,} | pos_weight: {pos_weight.item():.4f}")

optimizer = torch.optim.Adam(model.parameters(),lr = lr)
loss_fn = nn.BCELoss()


def train_one_epoch(model,loader):
    model.train()
    total_loss = 0.0
    n = 0
    for x,y in tqdm(loader,desc="Training_process",leave=False):
        x = x.to(device)
        y = y.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred,y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        n += x.size(0)
    return (total_loss)/ max(1,n)
@torch.inference_mode()
def eval_one_epoch(model,loader,loss_fn):
    model.eval()
    all_p = []
    all_y = []
    n = 0
    total_loss = 0.0
    for x,y in tqdm(loader,desc="Validation_process",leave=False):
        x = x.to(device)
        y = y.float().unsqueeze(1).to(device)
        prob = model(x)
        loss = loss_fn(prob,y)
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        all_p.append(prob.cpu().numpy())
        all_y.append(y.cpu().numpy())
    avg_loss = total_loss / max(1,n)
    y_prob = np.concatenate(all_p).ravel()
    y_true = np.concatenate(all_y).ravel()
    y_pred = (y_prob > 0.5).astype(int)

    return {
        'loss_test':    avg_loss,
        'acc':     accuracy_score(y_true, y_pred),
        'prec':    precision_score(y_true, y_pred, zero_division=0),
        'rec':     recall_score(y_true, y_pred, zero_division=0),
        'f1':      f1_score(y_true, y_pred, zero_division=0),
        'auc_roc': roc_auc_score(y_true, y_prob),
        'auc_pr':  average_precision_score(y_true, y_prob),
    }, y_true, y_prob
best_f1 = 0.0
history = []
for epoch in range(1, num_epochs + 1):
    start_time = time.time()
    train_loss    = train_one_epoch(model, train_loader)
    end_time = time.time()
    metrics, _, _ = eval_one_epoch(model, val_loader,loss_fn)
    history.append({'epoch': epoch, 'train_loss': train_loss, **metrics})

    print(f"Epoch {epoch:02d}/{num_epochs} | "
          f"loss_train={train_loss:.4f} | "
          f"loss_test={metrics['loss_test']:.4f}| "
          f"accuracy={metrics['acc']:.4f} | "
          f"precision={metrics['prec']:.4f}| "
          f"recall={metrics['rec']:.4f}| "
          f"f1={metrics['f1']:.4f} | "
          f"auc_roc={metrics['auc_roc']:.4f} |"
          f"time_per_epoch={(end_time-start_time):.4f}")
    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_f1': best_f1,
            'metrics': metrics
        }
        torch.save(checkpoint, 'bilbo.pt_ver2')



Benign: 730,168 | DGA: 256,148 | pos_weight: 3.4207


Epoch 01/10 | loss_train=0.1853 | loss_test=0.1343| accuracy=0.9523 | precision=0.9691| recall=0.8430| f1=0.9017 | auc_roc=0.9792 |time_per_epoch=130.9391


Epoch 02/10 | loss_train=0.1234 | loss_test=0.1109| accuracy=0.9623 | precision=0.9506| recall=0.9016| f1=0.9254 | auc_roc=0.9848 |time_per_epoch=129.0423


Epoch 03/10 | loss_train=0.1057 | loss_test=0.1072| accuracy=0.9621 | precision=0.9771| recall=0.8744| f1=0.9229 | auc_roc=0.9868 |time_per_epoch=127.9315


Epoch 04/10 | loss_train=0.0966 | loss_test=0.0997| accuracy=0.9662 | precision=0.9456| recall=0.9229| f1=0.9341 | auc_roc=0.9881 |time_per_epoch=128.0619


...

Epoch 06/10 | loss_train=0.0874 | loss_test=0.0944| accuracy=0.9677 | precision=0.9571| recall=0.9169| f1=0.9365 | auc_roc=0.9886 |time_per_epoch=130.1080


Epoch 07/10 | loss_train=0.0839 | loss_test=0.0982| accuracy=0.9680 | precision=0.9554| recall=0.9196| f1=0.9372 | auc_roc=0.9887 |time_per_epoch=128.2877


Epoch 08/10 | loss_train=0.0806 | loss_test=0.0926| accuracy=0.9688 | precision=0.9681| recall=0.9098| f1=0.9380 | auc_roc=0.9894 |time_per_epoch=128.4775


Epoch 09/10 | loss_train=0.0786 | loss_test=0.0963| accuracy=0.9687 | precision=0.9561| recall=0.9219| f1=0.9387 | auc_roc=0.9896 |time_per_epoch=128.0981


Epoch 10/10 | loss_train=0.0763 | loss_test=0.0948| accuracy=0.9691 | precision=0.9649| recall=0.9143| f1=0.9389 | auc_roc=0.9891 |time_per_epoch=128.8461


In [15]:
model = Hybrid_CNN_LSTM()
ckpt = torch.load('bilbo.pt_ver2', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)
print(f"Loaded epoch {ckpt['epoch']} | val F1: {ckpt['best_f1']:.4f}")

test_metrics, _, _ = eval_one_epoch(model, test_loader,loss_fn)
print("\n" + "="*50 + "\nTEST SET RESULTS\n" + "="*50)
for k, v in test_metrics.items():
    print(f"{k.upper():<10}: {v:.4f}")
print("-"*50)

Loaded epoch 10 | val F1: 0.9389



TEST SET RESULTS
LOSS_TEST : 0.0963
ACC       : 0.9690
PREC      : 0.9633
REC       : 0.9155
F1        : 0.9388
AUC_ROC   : 0.9893
AUC_PR    : 0.9814
--------------------------------------------------
